# Store Sales Time-Series Forecasting — CRISP-ML(Q) Pipeline

Corporación Favorita grocery sales (Ecuador) — the Kaggle competition.

**Dataset:** `data/store-sales-time-series-forecasting/` — 6 CSVs joined.
- `train.csv` — 3,000,888 daily rows · 2013-01-01 → 2017-08-15
- `stores.csv` — 54 stores (city, state, type, cluster)
- `transactions.csv` — daily transaction count per store
- `oil.csv` — daily WTI oil price (Ecuador economy ≈ oil price)
- `holidays_events.csv` — national/regional/local holidays
- `test.csv` — 28k rows for Kaggle submission (we use a local holdout instead)

**Target:** `sales` per `(date, store_nbr, family)`.
**Grain:** daily · 54 stores × 33 families = 1,782 series · 4.5 years history.
**Kaggle metric:** RMSLE.
**Method:** CRISP-ML(Q) — six phases, same playbook as the Inventory + Sales + Food labs.

> Lessons applied from prior labs:
> - **Time-based split** (last 30 days = holdout), no shuffle
> - **Lag + rolling features** grouped by (store, family)
> - **External regressors merged in**: oil price (ffill weekends), transactions, holiday flags
> - **Per-family routing** (33 dedicated LightGBMs) — the Sales lab winner pattern
> - **Tier 1 + Tier 2 benchmarks** with same defensive guards as Inventory
> - 3M rows requires LightGBM-grade scaling — RandomForest works but is the slowest; ExtraTrees skipped (memory)


In [ ]:
# Setup & Imports
import warnings; warnings.filterwarnings('ignore')
import os, sys, time, json
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.compose import ColumnTransformer
from sklearn.ensemble import (
    HistGradientBoostingRegressor, RandomForestRegressor, ExtraTreesRegressor,
)
from sklearn.impute import SimpleImputer
from sklearn.metrics import mean_absolute_error, mean_squared_error
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import (
    StandardScaler, MinMaxScaler, OneHotEncoder, PowerTransformer,
)

try:
    from lightgbm import LGBMRegressor
    HAS_LGBM = True
except ImportError:
    HAS_LGBM = False

# Import disk-checkpointing helper — works whether CWD is repo root or notebooks/
_HERE = Path('.').resolve()
for _candidate in (_HERE / 'notebooks', _HERE, _HERE.parent / 'notebooks'):
    if (_candidate / 'notebook_utils.py').exists():
        sys.path.insert(0, str(_candidate))
        break
from notebook_utils import cached, clear_cache

# Pretty defaults
sns.set_theme(style='whitegrid')
plt.rcParams['figure.figsize'] = (11, 5)
plt.rcParams['axes.titleweight'] = 'bold'
GOLD, INK, INK_SOFT = '#C9A86A', '#1A1D23', '#5E757D'

print(f'Python {sys.version.split()[0]}  ·  pandas {pd.__version__}  ·  LightGBM={HAS_LGBM}')


---
## Phase 1 — Business Understanding

### 1.1 The business question
*"How many units of each `(store, family)` will sell on each future day?"*

Operationally drives:
- **Replenishment** — daily case-pack orders to distribution centers
- **Promotion lift estimation** — `onpromotion` × baseline = expected uplift
- **Stockout prevention** for perishables (the gold/coral chart from the Sales lab)

### 1.2 Aspirational target
- **Kaggle leaderboard winners** sit at **RMSLE ≈ 0.36-0.42**
- Our target: a **single notebook end-to-end RMSLE < 0.50** with our toolkit (no kaggle-engineering hacks)
- **Stretch:** match top-100 (~RMSLE 0.42) via per-family LightGBM + tight features


---
## Phase 2 — Data Understanding

### 2.1 Load + merge all six CSVs


In [ ]:
ROOT = Path('.').resolve()
DATA_DIR = ROOT.parent / 'data' / 'store-sales-time-series-forecasting' if ROOT.name == 'notebooks' else ROOT / 'data' / 'store-sales-time-series-forecasting'

train_raw    = pd.read_csv(DATA_DIR / 'train.csv', parse_dates=['date'])
stores       = pd.read_csv(DATA_DIR / 'stores.csv')
transactions = pd.read_csv(DATA_DIR / 'transactions.csv', parse_dates=['date'])
oil          = pd.read_csv(DATA_DIR / 'oil.csv', parse_dates=['date'])
holidays     = pd.read_csv(DATA_DIR / 'holidays_events.csv', parse_dates=['date'])

df = (train_raw
      .merge(stores, on='store_nbr', how='left')
      .merge(transactions, on=['date','store_nbr'], how='left')
      .merge(oil, on='date', how='left'))

# Collapse national non-transferred holidays into a single flag + type
hol = holidays[(holidays['transferred']==False) & (holidays['locale']=='National')][['date','type']].drop_duplicates('date')
hol = hol.rename(columns={'type':'holiday_type'})
df = df.merge(hol, on='date', how='left')
df['is_holiday']  = df['holiday_type'].notna().astype(int)
df['holiday_type'] = df['holiday_type'].fillna('None')

# Oil prices not published on weekends — ffill then bfill
df['dcoilwtico']   = df['dcoilwtico'].ffill().bfill()
df['transactions'] = df['transactions'].fillna(0)

print(f'After merge: {len(df):,} rows · {df.shape[1]} cols')
print(f'Date range:  {df["date"].min().date()} → {df["date"].max().date()}')
print(f'Stores: {df["store_nbr"].nunique()}  ·  Families: {df["family"].nunique()}')
print(f'Series: ~{df["store_nbr"].nunique()*df["family"].nunique():,}')


### 2.2 Target distribution + zero rate

In [ ]:
print(f'sales summary:')
print(df['sales'].describe().round(2).to_string())
print(f'\nFraction of exact zeros: {(df["sales"]==0).mean()*100:.2f}%')
print(f'Fraction of small (<1):  {(df["sales"]<1).mean()*100:.2f}%')

fig, axes = plt.subplots(1, 2, figsize=(13, 4))
axes[0].hist(df['sales'].clip(upper=df['sales'].quantile(0.99)), bins=80, color=GOLD, edgecolor='white', linewidth=0.5)
axes[0].set_xlabel('sales (clipped at p99)'); axes[0].set_title('sales — raw (p0-p99)')
axes[1].hist(np.log1p(df['sales']), bins=80, color=INK, edgecolor='white', linewidth=0.5, alpha=0.85)
axes[1].set_xlabel('log1p(sales)'); axes[1].set_title('sales — log1p (Kaggle metric is RMSLE)')
plt.tight_layout(); plt.show()


### 2.3 Within-group autocorrelation — should lags help?

In [ ]:
rng = np.random.RandomState(42)
keys = df[['store_nbr','family']].drop_duplicates()
sampled = keys.iloc[rng.choice(len(keys), size=min(200, len(keys)), replace=False)]
autocorrs = []
for _, row in sampled.iterrows():
    s = df[(df['store_nbr']==row['store_nbr']) & (df['family']==row['family'])].sort_values('date')['sales'].values
    if len(s) >= 60:
        ac = np.corrcoef(s[:-1], s[1:])[0,1]
        if not np.isnan(ac): autocorrs.append(ac)
print(f'Median lag-1 autocorr: {np.median(autocorrs):+.3f}  ·  Mean: {np.mean(autocorrs):+.3f}')

plt.figure(figsize=(9, 3.5))
plt.hist(autocorrs, bins=30, color=GOLD, edgecolor='white')
plt.axvline(0, color='red', linestyle='--', alpha=0.6)
plt.title(f'Within-group lag-1 autocorrelation (n={len(autocorrs)})')
plt.tight_layout(); plt.show()


#### Insight — Autocorrelation
Strong positive autocorr (lag-1 typically 0.5-0.8). Lags + week-over-week (`lag_7`) should dominate.

---
## Phase 3 — Data Preparation

### 3.1 Calendar features + lag/rolling per (store, family)


In [ ]:
def add_features(df):
    df = df.sort_values(['store_nbr','family','date']).reset_index(drop=True)
    # Calendar
    df['year']        = df['date'].dt.year
    df['month']       = df['date'].dt.month
    df['day']         = df['date'].dt.day
    df['dayofweek']   = df['date'].dt.dayofweek
    df['weekofyear'] = df['date'].dt.isocalendar().week.astype(int)
    df['quarter']     = df['date'].dt.quarter
    df['is_weekend']  = df['dayofweek'].isin([5,6]).astype(int)
    df['is_payday']   = df['day'].isin([15,30,31]).astype(int)
    df['dayofyear']   = df['date'].dt.dayofyear
    df['sin_year']    = np.sin(2*np.pi*df['dayofyear']/365.25)
    df['cos_year']    = np.cos(2*np.pi*df['dayofyear']/365.25)
    # Lags + rolling per (store, family)
    g = df.groupby(['store_nbr','family'], sort=False)['sales']
    for lag in [1, 7, 14, 28]:
        df[f'sales_lag_{lag}'] = g.shift(lag)
    grp_ng = df.groupby(['store_nbr','family']).ngroup().values
    for w in [7, 14, 28]:
        shifted = g.shift(1)
        df[f'sales_roll_mean_{w}'] = shifted.groupby(grp_ng).rolling(w, min_periods=1).mean().reset_index(level=0, drop=True)
        df[f'sales_roll_std_{w}']  = shifted.groupby(grp_ng).rolling(w, min_periods=2).std().reset_index(level=0, drop=True)
    return df

df_model = add_features(df)
print(f'After feature engineering: {df_model.shape}')


### 3.2 Feature lists — Stage 1 vs Stage 2

In [ ]:
NUM_STANDARD = ['transactions','dcoilwtico','onpromotion',
                'sales_lag_1','sales_lag_7','sales_lag_14','sales_lag_28',
                'sales_roll_mean_7','sales_roll_std_7',
                'sales_roll_mean_14','sales_roll_std_14',
                'sales_roll_mean_28','sales_roll_std_28',
                'cluster']
NUM_FOURIER  = ['sin_year','cos_year']
NUM_MINMAX   = ['year','month','day','dayofweek','weekofyear','quarter','dayofyear']
BINARY       = ['is_weekend','is_payday','is_holiday']
CATEGORICAL  = ['family','city','state','type','holiday_type']

FEATURE_COLS_STAGE2 = NUM_STANDARD + NUM_MINMAX + NUM_FOURIER + BINARY + CATEGORICAL
FEATURE_COLS_STAGE1 = [c for c in FEATURE_COLS_STAGE2 if 'lag_' not in c and 'roll_' not in c]
print(f'Stage 2 features: {len(FEATURE_COLS_STAGE2)}  ·  Stage 1: {len(FEATURE_COLS_STAGE1)}')


### 3.3 Time-based split — last 30 days = holdout

In [ ]:
SPLIT_DATE = df_model['date'].max() - pd.Timedelta(days=30)
train_df = df_model[df_model['date'] <= SPLIT_DATE].copy().reset_index(drop=True)
test_df  = df_model[df_model['date'] >  SPLIT_DATE].copy().reset_index(drop=True)

X_train_s2 = train_df[FEATURE_COLS_STAGE2]; y_train = train_df['sales']
X_test_s2  = test_df[FEATURE_COLS_STAGE2];  y_test  = test_df['sales']
X_train_s1 = train_df[FEATURE_COLS_STAGE1]; X_test_s1 = test_df[FEATURE_COLS_STAGE1]

print(f'SPLIT_DATE   : {SPLIT_DATE.date()}')
print(f'Train rows   : {len(train_df):,}  ({train_df["date"].min().date()} → {train_df["date"].max().date()})')
print(f'Holdout rows : {len(test_df):,}  ({test_df["date"].min().date()} → {test_df["date"].max().date()})')


### 3.4 ColumnTransformer

In [ ]:
def build_preprocessor(feature_cols):
    num_std = [c for c in NUM_STANDARD if c in feature_cols]
    num_mm  = [c for c in NUM_MINMAX   if c in feature_cols]
    num_f   = [c for c in NUM_FOURIER  if c in feature_cols]
    cat     = [c for c in CATEGORICAL  if c in feature_cols]
    binary  = [c for c in BINARY       if c in feature_cols]
    return ColumnTransformer([
        ('std',  Pipeline([('imp', SimpleImputer(strategy='median')), ('sc', StandardScaler())]), num_std),
        ('mm',   Pipeline([('imp', SimpleImputer(strategy='median')), ('sc', MinMaxScaler())]), num_mm),
        ('pass', Pipeline([('imp', SimpleImputer(strategy='median'))]), num_f + binary),
        ('cat',  Pipeline([('imp', SimpleImputer(strategy='most_frequent')),
                            ('ohe', OneHotEncoder(handle_unknown='ignore', sparse_output=False))]), cat),
    ], remainder='drop')

prep_s1 = build_preprocessor(FEATURE_COLS_STAGE1)
prep_s2 = build_preprocessor(FEATURE_COLS_STAGE2)


---
## Phase 4 — Modeling

### 4.0 Metric helpers

In [ ]:
# Metric definitions — same convention as Inventory + Sales labs
def smape(y_true, y_pred):
    y_true, y_pred = np.asarray(y_true, float), np.asarray(y_pred, float)
    denom = (np.abs(y_true) + np.abs(y_pred)).clip(min=1e-6)
    return float(np.mean(2 * np.abs(y_pred - y_true) / denom) * 100)

def rmsle(y_true, y_pred):
    y_true = np.maximum(np.asarray(y_true, float), 0)
    y_pred = np.maximum(np.asarray(y_pred, float), 0)
    return float(np.sqrt(np.mean((np.log1p(y_pred) - np.log1p(y_true)) ** 2)))

def eval_all(name, y_true, y_pred):
    y_pred = np.clip(y_pred, 0, None)
    mae  = mean_absolute_error(y_true, y_pred)
    rmse = float(np.sqrt(mean_squared_error(y_true, y_pred)))
    sm   = smape(y_true, y_pred)
    rsl  = rmsle(y_true, y_pred)
    print(f'{name:<48s}  MAE={mae:8.2f}  RMSE={rmse:8.2f}  sMAPE={sm:5.1f}%  RMSLE={rsl:.4f}')
    return {'name': name, 'MAE': float(mae), 'RMSE': rmse, 'sMAPE': sm, 'RMSLE': rsl}

results = []


### 4.1 Baselines

In [ ]:
# Per-(store, family) historical mean
gm = train_df.groupby(['store_nbr','family'])['sales'].mean()
y_pred_gm = test_df.set_index(['store_nbr','family']).index.map(gm).to_numpy()
y_pred_gm = np.where(pd.isna(y_pred_gm), y_train.mean(), y_pred_gm).astype(float)
results.append(eval_all('Baseline: per-group mean', y_test, y_pred_gm))

# Naive lag-7 (week-over-week)
results.append(eval_all('Baseline: naive lag-7', y_test.values, X_test_s2['sales_lag_7'].fillna(y_train.mean()).values))

# Naive rolling-28
results.append(eval_all('Baseline: naive rolling-28', y_test.values, X_test_s2['sales_roll_mean_28'].fillna(y_train.mean()).values))


### 4.2 Stage 2 — LightGBM on the full feature set

In [ ]:
if HAS_LGBM:
    def fit_s2_lgbm():
        return Pipeline([('pre', prep_s2), ('m', LGBMRegressor(
            n_estimators=800, learning_rate=0.05, num_leaves=127,
            min_child_samples=20, subsample=0.85, colsample_bytree=0.85,
            random_state=42, n_jobs=-1, verbose=-1,
        ))]).fit(X_train_s2, y_train)
    pipe_s2_lgbm = cached('store_s2_lgbm', fit_s2_lgbm)
    results.append(eval_all('Stage 2: LightGBM', y_test, pipe_s2_lgbm.predict(X_test_s2)))


### 4.3 Stage 1 — LightGBM cold-start (no lags)

In [ ]:
if HAS_LGBM:
    def fit_s1_lgbm():
        return Pipeline([('pre', prep_s1), ('m', LGBMRegressor(
            n_estimators=600, learning_rate=0.05, num_leaves=127,
            random_state=42, n_jobs=-1, verbose=-1,
        ))]).fit(X_train_s1, y_train)
    pipe_s1_lgbm = cached('store_s1_lgbm', fit_s1_lgbm)
    results.append(eval_all('Stage 1: LightGBM (no lags)', y_test, pipe_s1_lgbm.predict(X_test_s1)))


### 4.4 Per-family routing — 33 LightGBMs (Sales-lab winner pattern)

In [ ]:
if HAS_LGBM:
    def fit_per_family():
        per_fam = {}
        for fam, tr_grp in train_df.groupby('family'):
            p = Pipeline([('pre', prep_s2), ('m', LGBMRegressor(
                n_estimators=400, learning_rate=0.05, num_leaves=63,
                random_state=42, n_jobs=-1, verbose=-1,
            ))])
            p.fit(tr_grp[FEATURE_COLS_STAGE2], tr_grp['sales'])
            per_fam[fam] = p
        return per_fam

    pipes_per_family = cached('store_per_family_lgbm', fit_per_family)
    all_preds = np.zeros(len(test_df), dtype=float)
    for fam, p in pipes_per_family.items():
        mask = (test_df['family'] == fam).values
        if mask.any():
            all_preds[mask] = p.predict(test_df.loc[mask, FEATURE_COLS_STAGE2])
    results.append(eval_all('Per-family LightGBM (routed)', y_test, all_preds))


### 4.5 Tier 1 — Additional benchmarks

In [ ]:
# 4.5.1 — CatBoost (native cats — fits 3M rows in 2-5 min)
try:
    from catboost import CatBoostRegressor
    CB_CAT_COLS = [c for c in CATEGORICAL if c in FEATURE_COLS_STAGE2]
    def fit_catboost():
        Xtr = X_train_s2.copy(); Xte = X_test_s2.copy()
        for c in CB_CAT_COLS:
            Xtr[c] = Xtr[c].astype(str); Xte[c] = Xte[c].astype(str)
        m = CatBoostRegressor(iterations=600, learning_rate=0.05, depth=7,
                              loss_function='MAE', cat_features=CB_CAT_COLS,
                              nan_mode='Min', random_seed=42, verbose=False)
        m.fit(Xtr, y_train); return m, Xte
    cb_model, _Xte_cb = cached('store_catboost', fit_catboost)
    results.append(eval_all('Tier 1: CatBoost', y_test, cb_model.predict(_Xte_cb)))
except ImportError:
    print('catboost not installed')


In [ ]:
# 4.5.2 — HistGradientBoosting
def fit_histgb():
    return Pipeline([('pre', prep_s2), ('m', HistGradientBoostingRegressor(
        max_iter=600, learning_rate=0.05, max_leaf_nodes=127, min_samples_leaf=20,
        l2_regularization=1.0, random_state=42,
    ))]).fit(X_train_s2, y_train)
pipe_hgb = cached('store_histgb', fit_histgb)
results.append(eval_all('Tier 1: HistGradientBoosting', y_test, pipe_hgb.predict(X_test_s2)))


### 4.6 Tier 2 — Modern DL forecasters (best-effort)

> 3M rows × multi-series — these can take 30-90 min each. Run only if you have the time. Defensive guards as before.

In [ ]:
# 4.6.0 — NF setup (with Mac/OMP guards)
import os
os.environ['KMP_DUPLICATE_LIB_OK'] = 'TRUE'
os.environ['PYTORCH_ENABLE_MPS_FALLBACK'] = '1'
try:
    import torch
    torch.backends.mps.is_available = lambda: False
    torch.backends.mps.is_built     = lambda: False
    import logging
    for _n in ('pytorch_lightning','lightning','lightning.pytorch'):
        logging.getLogger(_n).setLevel(logging.ERROR)
    from neuralforecast import NeuralForecast
    from neuralforecast.models import NHITS, TFT
    from neuralforecast.losses.pytorch import MAE as NFMAE
    _HAS_NF = True
except Exception as _e:
    _HAS_NF = False
    print(f'neuralforecast unavailable — Tier 2 will skip. ({type(_e).__name__})')

if _HAS_NF:
    df_long_nf = train_df[['store_nbr','family','date']].copy()
    df_long_nf['unique_id'] = df_long_nf['store_nbr'].astype(str) + '_' + df_long_nf['family']
    df_long_nf['ds'] = df_long_nf['date']
    df_long_nf['y']  = train_df['sales'].values
    df_long_nf = df_long_nf[['unique_id','ds','y']]
    H_nf = test_df['date'].nunique()

    def _merge_nf(fcst, col):
        t = test_df.copy()
        t['unique_id'] = t['store_nbr'].astype(str) + '_' + t['family']
        t['ds'] = t['date']
        m = t.merge(fcst, on=['unique_id','ds'], how='left')
        return np.maximum(m[col].fillna(y_train.mean()).values, 0)

    def _safe_nf_fit(name, factory):
        try:
            def _fit():
                nf = NeuralForecast(models=[factory()], freq='D')
                nf.fit(df=df_long_nf)
                return nf.predict()
            fcst = cached(f'store_{name}', _fit)
            col = name.upper() if name.upper() in fcst.columns else [c for c in fcst.columns if c not in ('unique_id','ds')][0]
            return eval_all(f'Tier 2: {name.upper()} (Nixtla)', y_test, _merge_nf(fcst, col))
        except Exception as e:
            print(f'  [skipped — {name}] {type(e).__name__}: {str(e)[:120]}')
            return None
    print(f'NF ready · series={df_long_nf.unique_id.nunique()} · H={H_nf}')


In [ ]:
# 4.6.1 — NHITS
if _HAS_NF:
    r = _safe_nf_fit('nhits', lambda: NHITS(h=H_nf, input_size=28, loss=NFMAE(),
                                            max_steps=300, batch_size=64, random_seed=42, accelerator='cpu'))
    if r: results.append(r)


In [ ]:
# 4.6.2 — TFT
if _HAS_NF:
    r = _safe_nf_fit('tft', lambda: TFT(h=H_nf, input_size=28, hidden_size=64, n_head=4,
                                        max_steps=300, batch_size=32, random_seed=42, accelerator='cpu'))
    if r: results.append(r)


---
## Phase 5 — Evaluation

### 5.1 Final leaderboard (sorted by RMSLE — the Kaggle metric)

In [ ]:
leaderboard = pd.DataFrame(results).sort_values('RMSLE').reset_index(drop=True)
leaderboard

### 5.2 Per-family RMSLE — which categories miss?

In [ ]:
if HAS_LGBM:
    final_pred = all_preds  # per-family routed
    label = 'Per-family LightGBM (routed)'
else:
    final_pred = pipe_hgb.predict(X_test_s2)
    label = 'Tier 1: HistGradientBoosting'

per_fam = (
    pd.DataFrame({'family': test_df['family'].values,
                  'abs_err': np.abs(y_test.values - np.clip(final_pred, 0, None)),
                  'rmsle_term': (np.log1p(np.clip(final_pred,0,None)) - np.log1p(np.maximum(y_test.values,0)))**2})
    .groupby('family').agg(MAE=('abs_err','mean'), n=('abs_err','count'),
                            RMSLE=('rmsle_term', lambda x: float(np.sqrt(x.mean())))).round(4)
    .sort_values('RMSLE')
)
print(f'Using: {label}\n')
print(per_fam.to_string())


---
## Phase 6 — Deployment

### 6.1 Save winning artifact + metadata

In [ ]:
MODEL_DIR = ROOT.parent / 'model' / 'store_sales' if ROOT.name == 'notebooks' else ROOT / 'model' / 'store_sales'
MODEL_DIR.mkdir(parents=True, exist_ok=True)

import joblib
winner_row = leaderboard.iloc[0]
winner_name = winner_row['name']
print(f'Winner: {winner_name}  ·  RMSLE={winner_row["RMSLE"]:.4f}  ·  MAE={winner_row["MAE"]:.2f}')

if 'Per-family' in winner_name and HAS_LGBM:
    joblib.dump(pipes_per_family, MODEL_DIR / 'best_model_per_family.pkl')
    metadata = {
        'model_name': winner_name, 'feature_columns': FEATURE_COLS_STAGE2,
        'mae': float(winner_row['MAE']), 'rmsle': float(winner_row['RMSLE']),
        'routing_key': 'family', 'all_results': results,
    }
elif HAS_LGBM:
    joblib.dump(pipe_s2_lgbm, MODEL_DIR / 'best_model.pkl')
    metadata = {
        'model_name': winner_name, 'feature_columns': FEATURE_COLS_STAGE2,
        'mae': float(winner_row['MAE']), 'rmsle': float(winner_row['RMSLE']),
        'all_results': results,
    }
else:
    joblib.dump(pipe_hgb, MODEL_DIR / 'best_model.pkl')
    metadata = {'model_name': winner_name, 'feature_columns': FEATURE_COLS_STAGE2,
                'mae': float(winner_row['MAE']), 'rmsle': float(winner_row['RMSLE']),
                'all_results': results}

joblib.dump(metadata, MODEL_DIR / 'model_metadata.pkl')
print(f'Saved → {MODEL_DIR}')
